# Thí nghiệm 1: Ước lượng hiệu quả thuốc (Estimation)
### Mục tiêu: 
So sánh không Resampling, Jackknife, và Bootstrap

### Kì vọng kết quả
1. **Naive Estimate**: Sẽ cho ra một giá trị đơn lẻ cho Mean và Median từ mẫu $n=50$, nhưng không cung cấp thông tin về độ tin cậy (Standard Error) nếu không áp dụng các công thức lý thuyết phức tạp.
2. **Jackknife**: Kỳ vọng sẽ ước lượng SE cho Mean khá chính xác. Tuy nhiên, đối với Median, Jackknife có thể gặp khó khăn và cho ra kết quả không ổn định do Median là một chỉ số không trơn (non-smooth).
3. **Bootstrap**: Kỳ vọng sẽ cung cấp ước lượng SE và phân phối thực nghiệm mạnh mẽ cho cả Mean và Median. Phân phối Bootstrap của Median sẽ phản ánh tốt hơn sự biến thiên của quần thể so với Jackknife.

### Mô tả dataset: Medical Cost Personal dataset
- **Nguồn**: Insurance Forecast 
- **Đặc điểm**: Dữ liệu cho biết mối liên hệ giuawxcacs đặc điểm như tuổi, số con, khu vực ... và chi phí y tế cá nhân tại Mỹ

Chi tiết các cột trong Dataset: 

| Cột | Ý nghĩa | Ghi chú |
|:---|:---|:---|
| **age** | Tuổi của người thụ hưởng bảo hiểm | Biến định lượng |
| **sex** | Giới tính | male/female |
| **bmi** | Chỉ số khối cơ thể (Body Mass Index) | weight/height^2 |
| **children** | Số lượng trẻ em/người phụ thuộc | Biến định lượng |
| **smoker** | Tình trạng hút thuốc | yes/no (Yếu tố ảnh hưởng mạnh nhất) |
| **region** | Khu vực sinh sống tại Mỹ | northeast, southeast, southwest, northwest |
| **charges** | Chi phí y tế cá nhân | **Biến mục tiêu (Target variable)** |

### Các chỉ số đo lường (Metrics):

Trong thí nghiệm này, chúng ta tập trung vào các đại lượng thống kê sau:
1. **Standard Error (SE)**: Sai số chuẩn của ước lượng, đo lường độ biến động của estimator qua các lần resampling. SE càng nhỏ, ước lượng càng ổn định.
2. **Confidence Interval (CI)**: Khoảng tin cậy 95%. Cho chúng ta một phạm vi giá trị mà tham số thực của quần thể có khả năng nằm trong đó.
3. **Bias (Độ chệch)**: Sự sai khác giữa giá trị trung bình của các ước lượng từ resampling và giá trị thực tế trên mẫu gốc.

In [4]:
# Bước 1: Tải dataset từ Google Drive hoặc local
import os, sys

# Cấu hình thư mục data
if 'google.colab' in sys.modules:
    data_dir = '/content/data'
else:
    data_dir = os.path.abspath('../data') if os.path.exists('../data') else os.path.abspath('data')
os.makedirs(data_dir, exist_ok=True)

path = os.path.join(data_dir, 'insurance.csv')

# Nếu chưa có file, download từ Google Drive
if not os.path.exists(path):
    try:
        import gdown
        # Cách lấy ID: Mở file ggdrive → Copy từ URL: drive.google.com/file/d/[ID_HERE]/view
        gdrive_id = "1x3wzT431BaRsyWHQf9Bdi6BaCVgHg1j9" 
        url = f"https://drive.google.com/uc?id={gdrive_id}"
        print(f"Downloading from Google Drive: {url}")
        gdown.download(url, path, quiet=False)
        print(f"Downloaded: {path}")
    except ImportError:
        print("gdown not found. Installing...")
        os.system("pip install gdown")
        import gdown
        gdrive_id = "1x3wzT431BaRsyWHQf9Bdi6BaCVgHg1j9"
        url = f"https://drive.google.com/uc?id={gdrive_id}"
        gdown.download(url, path, quiet=False)
        print(f"Downloaded: {path}")
else:
    print(f"File already exists: {path}")

File already exists: /content/data/insurance.csv


In [5]:
import pandas as pd

# Tải và hiển thị thông tin dataset
df = pd.read_csv(path)

print("> Dataset Info:")
print(f"  - Shape: {df.shape[0]} observations × {df.shape[1]} features")
print(f"\n  - First 5 rows:")
print(df.head())

print(f"\n>  - Thống kê cột 'charges' (Target variable):")
print(df['charges'].describe())

print(f"\nDataset loaded successfully!")

> Dataset Info:
  - Shape: 1338 observations × 7 features

  - First 5 rows:
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520

>  - Thống kê cột 'charges' (Target variable):
count     1338.000000
mean     13270.422265
std      12110.011237
min       1121.873900
25%       4740.287150
50%       9382.033000
75%      16639.912515
max      63770.428010
Name: charges, dtype: float64

Dataset loaded successfully!


## 1.2 Ba phương pháp ước lượng

| Phương pháp | Đặc điểm | Ưu điểm | Nhược điểm |
|---|---|---|---|
| **Naive (không resampling)** | Chỉ dùng sample statistic | Đơn giản | Không có CI, không biết bias/variance |
| **Jackknife** | Leave-one-out resampling | Ước lượng được bias, variance rõ | Chậm, không tốt với median |
| **Bootstrap** | Resample ngẫu nhiên với thay thế | Linh hoạt, mạnh mẽ | Cần nhiều samples |

### Công thức:

**Naive Mean**: $\hat{\mu}_{naive} = \frac{1}{n} \sum_{i=1}^{n} x_i$

**Jackknife SE**: $SE_{JK} = \sqrt{\frac{n-1}{n} \sum_{i=1}^{n} (\hat{\theta}_{(-i)} - \bar{\hat{\theta}})^2}$

**Bootstrap SE**: $SE_{Boot} = \sqrt{\frac{1}{B} \sum_{b=1}^{B} (\hat{\theta}^{*b} - \bar{\hat{\theta}^*})^2}$

In [6]:
import sys, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

# Thêm thư mục gốc vào path để import được modules và utils
sys.path.append(os.path.abspath('..'))

from modules.estimation import jackknife, bootstrap, run_full_estimation
from utils.metrics import calc_se, calc_ci


ModuleNotFoundError: No module named 'modules'

In [ ]:
# Bước 3: Chạy thí nghiệm chính
# Lấy mẫu 50 bệnh nhân từ cột chi phí (charges)
data = df['charges'].sample(50, random_state=1).values
print(f"> Sample size: {len(data)} bệnh nhân")
print(f"> Charges range: ${data.min():.2f} - ${data.max():.2f}")

# Tìm phân phối estimator, ước lượng SE, CI cho Mean
mean_jk_dist, mean_boot_dist, mean_jk_se, mean_boot_se, mean_jk_ci, mean_boot_ci = run_full_estimation(data, np.mean)

# Vẽ biểu đồ histogram cho mean_jk_dist, mean_boot_dist (ex1_mean_dist)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(mean_boot_dist, kde=True, ax=ax1, color='blue', alpha=0.6)
ax1.axvline(np.mean(data), color='red', linestyle='--', label=f'Original Mean: ${np.mean(data):.2f}')
ax1.set_title("Bootstrap Distribution (Mean)")
ax1.legend()

sns.histplot(mean_jk_dist, kde=True, ax=ax2, color='orange', alpha=0.6)
ax2.axvline(np.mean(data), color='red', linestyle='--', label=f'Original Mean: ${np.mean(data):.2f}')
ax2.set_title("Jackknife Distribution (Mean)")
ax2.legend()
plt.show()

# Tìm phân phối estimator, ước lượng SE, CI cho Median (ex1_median_dist)
median_jk_dist, median_boot_dist, median_jk_se, median_boot_se, median_jk_ci, median_boot_ci = run_full_estimation(data, np.median)

# Vẽ biểu đồ histogram cho median_jk_dist, median_boot_dist
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(median_boot_dist, kde=True, ax=ax1, color='blue', alpha=0.6)
ax1.axvline(np.median(data), color='red', linestyle='--', label=f'Original Median: ${np.median(data):.2f}')
ax1.set_title("Bootstrap Distribution (Median)")
ax1.legend()

sns.histplot(median_jk_dist, kde=True, ax=ax2, color='red', alpha=0.6)
ax2.axvline(np.median(data), color='red', linestyle='--', label=f'Original Median: ${np.median(data):.2f}')
ax2.set_title("Jackknife Distribution (Median)")
ax2.legend()
plt.show()

### Nhận xét về phân phối estimator:
1. **Đối với Mean**: 
   - Cả phân phối Bootstrap và Jackknife đều có dạng hình chuông (xấp xỉ phân phối chuẩn), tập trung quanh giá trị Mean của mẫu gốc.
   - Điều này cho thấy Mean là một chỉ số "trơn" (smooth), ít nhạy cảm với việc thay đổi nhỏ trong tập dữ liệu.
2. **Đối với Median**:
   - **Bootstrap**: Cho ra một phân phối tương đối đầy đủ, phản ánh tốt các khả năng biến thiên của trung vị.
   - **Jackknife**: Phân phối trông rất "rời rạc" hoặc có ít giá trị khác nhau. Điều này là do khi loại bỏ 1 điểm khỏi mẫu $n=50$, Median chỉ có thể nhận một vài giá trị lân cận, không tạo ra được sự biến thiên liên tục như Bootstrap. 
   - Đây chính là lý do Jackknife không hiệu quả để ước lượng sai số cho các chỉ số không trơn như Median.

In [ ]:
# print giá trị CI, SE của Mean, median

# Vẽ biểu đồ so sánh, có thể sửa ccacs biến cho khớp
sns.set_theme(style='whitegrid', context='talk')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Biểu đồ: So sánh Bootstrap vs Jackknife SE cho Mean (ex1_mean_SE)
ax1.bar(['Bootstrap', 'Jackknife'], [mean_boot_se, mean_jk_se], color=['blue', 'orange'])
ax1.set_title('Mean Estimation - Standard Error Comparison', fontsize=14, fontweight='bold')
ax1.set_ylabel('Standard Error (USD)', fontsize=12)
ax1.grid(axis='y', alpha=0.3)

# Biểu đồ 2: So sánh Bootstrap vs Jackknife SE cho Median  (ex1_median_se)
ax2.bar(['Bootstrap', 'Jackknife'], [median_boot_se, median_jk_se], color=['blue', 'red'])
ax2.set_title('Median Estimation - Standard Error Comparison', fontsize=14, fontweight='bold')
ax2.set_ylabel('Standard Error (USD)', fontsize=12)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Phân tích sai số chuẩn (Standard Error - SE):
1. **Đối với Mean**:
   - Giá trị SE từ Jackknife và Bootstrap thường rất sát nhau. Điều này khẳng định rằng với các thống kê có tính chất trơn, cả hai phương pháp resampling đều cho hiệu quả tương đương và đáng tin cậy.
2. **Đối với Median**:
   - **Sự khác biệt rõ rệt**: Jackknife SE thường cho kết quả khác biệt (thường là nhỏ hơn hoặc biến động mạnh) so với Bootstrap SE.
   - **Lý do**: Khi loại bỏ lần lượt từng phần tử (Jackknife), tập hợp các giá trị Median thu được rất hạn chế, dẫn đến việc tính toán độ lệch chuẩn của các giá trị này không phản ánh đúng sai số thực tế của thống kê trung vị.
3. **Kết luận**: Bootstrap vượt trội hơn Jackknife trong việc ước lượng sai số cho các chỉ số nhạy cảm như Median. Trong thực tế phân tích chi phí y tế (vốn thường dùng Median để tránh ảnh hưởng của outlier), Bootstrap là công cụ tối ưu để đánh giá độ tin cậy của ước lượng.

In [ ]:
# Bước 4: Vẽ phân phối dữ liệu gốc (chưa biết làm gì nhma thấy tiêc nên giữ lại cell này =))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ histogram của dữ liệu gốc
axes[0].hist(data, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(data), color='blue', linestyle='--', linewidth=2, label=f'Mean: ${np.mean(data):.0f}')
axes[0].axvline(np.median(data), color='red', linestyle='--', linewidth=2, label=f'Median: ${np.median(data):.0f}')
axes[0].set_title('Distribution of Medical Charges (n=50)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Charges (USD)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Biểu đồ box plot để thấy outliers
axes[1].boxplot(data, vert=True)
axes[1].set_title('Box Plot: Detecting Outliers', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Charges (USD)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📈 Thống kê dữ liệu:")
print(f"  - Min: ${np.min(data):.2f}")
print(f"  - Q1: ${np.percentile(data, 25):.2f}")
print(f"  - Median: ${np.median(data):.2f}")
print(f"  - Q3: ${np.percentile(data, 75):.2f}")
print(f"  - Max: ${np.max(data):.2f}")
print(f"  - Skewness: {pd.Series(data).skew():.3f} (>0 = right-skewed)")

(chưa biết làm gì nhma thấy tiêc nên giữ lại cell này =)))
## 1.6 Visualize phân phối của dữ liệu

Hình dưới đây giúp hiểu **tại sao Bootstrap tốt hơn Jackknife cho Median**:
- Nếu phân phối có **outliers lớn** → Median sẽ biến động mạnh → Bootstrap capture tốt hơn
- Nếu phân phối **symmetric** → Mean và Median gần nhau → Cả 2 phương pháp OK

 (chưa biết làm gì nhma thấy tiêc nên giữ lại cell này =))

## 1.3 Thí nghiệm chính: So sánh Mean và Median estimation

### Case A: Ước lượng Mean (Trung bình)
- **Mean là gì**: Giá trị trung bình của dữ liệu - một smooth estimator
- **Kỳ vọng**: Vì mean là smooth, Jackknife và Bootstrap sẽ đều hoạt động tốt
- **Ý nghĩa**: Chi phí điều trị trung bình

### Case B: Ước lượng Median (Trung vị)
- **Median là gì**: Giá trị ở giữa dữ liệu - một non-smooth estimator
- **Kỳ vọng**: Jackknife sẽ hoạt động kém, Bootstrap sẽ tốt hơn
- **Ý nghĩa**: Chi phí điều trị "bình thường" (representative value)